### Datastax vectorDB

In [1]:
ASTRA_DB_API_ENDPOINT="https://6d9c1077-5606-415c-8937-46852d673dad-us-east-2.apps.astra.datastax.com"

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

d:\RAG System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 935.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
import os
from langchain_astradb import AstraDBVectorStore
os.environ["ASTRA_DB_APPLICATION_TOKEN"] = os.getenv("ASTRA_DB_APPLICATION_TOKEN")

In [11]:
vectorstore = AstraDBVectorStore(
    collection_name="rag_collection",
    embedding=embedding_model,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    namespace=None,
    token=os.getenv("ASTRA_DB_APPLICATION_TOKEN")

)
vectorstore

In [12]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
data_loader=TextLoader("../data/python.txt", encoding="utf-8")
documents=data_loader.load()
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
chunks=text_splitter.split_documents(documents)

In [13]:
vectorstore.add_documents(chunks)

['9930cdf7e733477cb792fd0b45b9b7de',
 'a9196f5e540f44a19ad56572f83d9146',
 '8216871b455f4b2f9594a50ac79e0628',
 '6e4c0ef0148248a886090cf25828af63',
 '6007a05ca1294eec9e8c1ba2e0b7ff38',
 '0be4780cecb44882832c328ce4d04a77']

In [15]:
search_results = vectorstore.similarity_search("What is Python?", k=3)
print(search_results)

[Document(id='9930cdf7e733477cb792fd0b45b9b7de', metadata={'source': '../data/python.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level programming language known for its simplicity and readability. \nIt was created by Guido van Rossum and released in 1991.'), Document(id='a9196f5e540f44a19ad56572f83d9146', metadata={'source': '../data/python.txt'}, page_content='Key Features:\n- Easy to learn and use\n- Large community support\n- Cross-platform compatibility\n\nPython is widely used in web development, data science, machine learning, and automation.'), Document(id='0be4780cecb44882832c328ce4d04a77', metadata={'source': '../data/python.txt'}, page_content='Key Steps:\n- Data collection\n- Data cleaning\n- Data analysis\n- Visualization\n\nTools used include Python, R, and SQL.')]
